# Tải, gán nhãn và chia 5 fold cho GSE68465, TCGA-LUAD, TCGA-LUSC

Notebook này gộp `geo_lung_download.ipynb` và `tcga_download.ipynb` thành một
luồng duy nhất:

1. Tải dữ liệu thô (GEO series matrix cho GSE68465, GDC API cho TCGA-LUAD/LUSC).
2. Gán nhãn giai đoạn và số hoá vào cột **`label`**
   (`Normal=0, I=1, II=2, III=3, IV=4` — cùng `STAGE_TO_CODE` với 2 notebook cũ).
3. Chuyển về **gene symbol**: mỗi cột biểu hiện mang đúng tên gene
   (GSE68465: log2 MAS5, probe → gene; TCGA: log2(TPM+1), chỉ gene `protein_coding`).
4. Chia dữ liệu thành **5 phần P1–P5 cân bằng lớp, theo nhóm bệnh nhân**
   (`StratifiedGroupKFold`), rồi xoay vòng thành **5 fold**: fold *r* có
   test = P*r*, val = P*r+1* (P5 → P1), train = gộp 3 phần còn lại (tỉ lệ 3/1/1).
   Mọi mẫu của cùng một bệnh nhân nằm trọn trong một phần, nên trong mỗi fold
   không bệnh nhân nào xuất hiện ở hai tập train/val/test (có kiểm tra `assert`).
5. Lưu 5 fold ở 2 định dạng (H5 và CSV), mỗi định dạng đóng 1 file zip
   → 3 bộ × 2 = **6 file zip**.

Cấu trúc mỗi zip: `<bo>/fold_<r>/{train,val,test}.<h5|csv>`. Mỗi file là bảng
`mẫu × (gene..., label)`, index là `sample_id`. Kèm thêm `fold_assignment.csv`
(sample_id, patient_id, stage, label, part, vai trò của mẫu trong từng fold)
và `label_mapping.json`.

## 1. Thiết lập

In [ ]:
import os, io, re, sys, json, time, gzip, tarfile, zipfile, warnings
import urllib.request
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/kltn_data"                          # du lieu tho (tam, local)
    OUTDIR = "/content/drive/MyDrive/KLTN/data/folds"       # 6 file zip ket qua
else:
    WORKDIR = os.path.abspath("kltn_data")
    OUTDIR = os.path.join(WORKDIR, "folds")

RAWDIR = os.path.join(WORKDIR, "raw")
os.makedirs(RAWDIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)

N_FOLDS = 5
# Xoay vong: fold r -> test = phan r, val = phan r+1 (5 -> 1), train = 3 phan con lai
FOLD_PLAN = {r: {"test": r, "val": r % N_FOLDS + 1,
                 "train": [p for p in range(1, N_FOLDS + 1) if p not in (r, r % N_FOLDS + 1)]}
             for r in range(1, N_FOLDS + 1)}
SEED_SEARCH = range(100)   # thu nhieu seed, chon cach chia can bang lop nhat

# GSE68465: 19 mau "Normal" thuc chat la RNA tham chieu Stratagene (source_name
# = 'Stratagene'), khong phai mo phoi thuong cua benh nhan. Notebook cu van giu
# chung voi nhan Normal -> mac dinh giu nguyen de nhat quan. Dat False de bo.
GSE_KEEP_REFERENCE_NORMAL = True

print("Colab:", IN_COLAB)
print("Du lieu tho:", RAWDIR)
print("Ket qua    :", OUTDIR)
for r, plan in FOLD_PLAN.items():
    print(f"  fold {r}: train = phan {plan['train']}, val = phan {plan['val']}, test = phan {plan['test']}")

In [ ]:
%pip install -q tables

## 2. Hàm dùng chung: chuẩn hoá nhãn giai đoạn

Giữ nguyên logic của 2 notebook cũ: đọc `Stage IIIB`, `stage: 2`, và mã pTNM
kiểu `pN0pT1` (GSE68465 chỉ ghi pT/pN, suy ra stage AJCC giả định M0).
Mẫu khối u không đọc được giai đoạn bị **loại**, không gán nhầm thành Normal.

In [ ]:
ARABIC_TO_ROMAN = {"1": "I", "2": "II", "3": "III", "4": "IV"}
MISSING = {"", "na", "n/a", "nan", "none", "null", "unknown",
           "not available", "not reported", "---", "-", "."}

STAGE_ORDER = ["Normal", "I", "II", "III", "IV"]
STAGE_TO_CODE = {s: i for i, s in enumerate(STAGE_ORDER)}


def stage_from_tnm_string(s):
    # Quy tac AJCC NSCLC, gia dinh M0 (cohort da phau thuat cat bo):
    #   N2/N3, hoac T4, hoac (T3 va N1) -> III ; T3 hoac N1 -> II ; con lai -> I
    t_match = re.search(r"p?t\s*([0-4])", s)
    n_match = re.search(r"p?n\s*([0-3])", s)
    if not t_match or not n_match:
        return None
    t, n = int(t_match.group(1)), int(n_match.group(1))
    if n >= 2 or t >= 4 or (t == 3 and n == 1):
        return "III"
    if t == 3 or n == 1:
        return "II"
    return "I"


def normalize_stage(x):
    # 'pathological stage: IIIB' -> 'III', 'disease_stage: pN0pT1' -> 'I'. None neu khong doc duoc.
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip().lower()
    if s in MISSING:
        return None
    # ma pTNM phai bat truoc, neu khong doan re.sub duoi se xoa mat pT/pN
    if re.search(r"p?t\s*[0-4]", s) and re.search(r"p?n\s*[0-3]", s):
        tnm_stage = stage_from_tnm_string(s)
        if tnm_stage is not None:
            return tnm_stage
    s = re.sub(r"^.*stage\s*[:=]?\s*", "", s).strip()
    if s in MISSING:
        return None
    if re.match(r"0(?![0-9])", s):
        return "0"
    m = re.match(r"(iv|iii|ii|i)(?![ivx])", s)   # khong dung \b vi "ia" se khong khop
    if m:
        return m.group(1).upper()
    m = re.match(r"([1-4])(?![0-9])", s)
    if m:
        return ARABIC_TO_ROMAN[m.group(1)]
    return None


def collapse_by_symbol(expr, symbols):
    # expr: feature x sample; symbols: Series cung index. Nhieu feature cung 1 gene
    # -> giu feature co phuong sai lon nhat (khong dung nhan, khong ro ri).
    sym = symbols.reindex(expr.index).dropna()
    e = expr.loc[sym.index]
    order = e.var(axis=1).sort_values(ascending=False).index
    e, sym = e.loc[order], sym.loc[order]
    keep = ~sym.duplicated().values
    e = e[keep]
    e.index = pd.Index(sym[keep].values, name="gene")
    return e.sort_index()


def build_table(expr_gene_x_sample, stage, patient_id):
    # Tra ve (data, meta). data: sample x (gene..., label), index = sample_id.
    X = expr_gene_x_sample.T.astype(np.float32)
    X.index.name = "sample_id"
    X.columns.name = None
    stage = stage.loc[X.index]
    data = X.copy()
    data["label"] = stage.map(STAGE_TO_CODE).astype(int).values
    meta = pd.DataFrame({"patient_id": patient_id.loc[X.index].values,
                         "stage": stage.values,
                         "label": data["label"].values}, index=X.index)
    return data, meta

## 3. GSE68465 (GEO)

Tải `series_matrix.txt.gz`, tách metadata và ma trận probe × mẫu.
GSE68465 không có mã bệnh nhân riêng: 443 khối u đến từ 443 bệnh nhân khác nhau,
nên `patient_id = GSM`. Probe được ánh xạ sang gene bằng file chú giải chính
thức `GPL96.annot.gz`; probe ánh xạ tới nhiều gene (`A///B`) bị bỏ vì không
xác định được tên gene duy nhất.

In [ ]:
GSE = "GSE68465"
GSE_RAW = os.path.join(RAWDIR, GSE)
os.makedirs(GSE_RAW, exist_ok=True)


def download(url, dest_dir):
    path = os.path.join(dest_dir, url.rsplit("/", 1)[-1])
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    urllib.request.urlretrieve(url, path)
    return path


def geo_stub(acc):
    # Thu muc FTP cua GEO: GSE68465 -> GSE68nnn, GPL96 -> GPLnnn
    prefix, num = acc[:3], acc[3:]
    return prefix + (num[:-3] if len(num) > 3 else "") + "nnn"


def matrix_file_urls(gse):
    stub = geo_stub(gse)
    base = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{stub}/{gse}/matrix/"
    with urllib.request.urlopen(base, timeout=120) as r:
        html = r.read().decode("utf-8", errors="replace")
    files = sorted(set(re.findall(r'href="([^"]*series_matrix\.txt\.gz)"', html)))
    if not files:
        raise RuntimeError(f"Khong tim thay series matrix cho {gse}")
    return [base + f for f in files]


def parse_series_matrix(path):
    # Tra ve (expr, pheno). expr: probe x sample. pheno: sample x metadata.
    meta, table_lines, in_table = {}, [], False
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n").rstrip("\r")
            if line.startswith("!series_matrix_table_begin"):
                in_table = True
                continue
            if line.startswith("!series_matrix_table_end"):
                in_table = False
                continue
            if in_table:
                if line:
                    table_lines.append(line)
            elif line.startswith("!Sample_"):
                parts = line.split("\t")
                meta.setdefault(parts[0][1:], []).append([v.strip().strip('"') for v in parts[1:]])

    # key lap lai (vd Sample_characteristics_ch1) -> danh so
    flat = {}
    for key, occ in meta.items():
        if len(occ) == 1:
            flat[key] = occ[0]
        else:
            for i, o in enumerate(occ, 1):
                flat[f"{key}_{i}"] = o
    pheno = pd.DataFrame(flat).set_index("Sample_geo_accession")
    pheno.index.name = "gsm"

    expr = pd.read_csv(io.StringIO("\n".join(table_lines)), sep="\t", index_col=0, low_memory=False)
    expr.index.name = "probe"
    expr = expr.apply(pd.to_numeric, errors="coerce")
    expr = expr.loc[:, [c for c in expr.columns if c in pheno.index]]
    return expr, pheno


exprs, phenos_gse = [], []
for url in matrix_file_urls(GSE):
    print("tai", url.rsplit("/", 1)[-1], "...", end=" ")
    e, p = parse_series_matrix(download(url, GSE_RAW))
    exprs.append(e)
    phenos_gse.append(p)
    print(f"{e.shape[0]} probe x {e.shape[1]} mau")

gse_expr = pd.concat(exprs, axis=1)
gse_pheno = pd.concat(phenos_gse, axis=0)
print("TONG:", gse_expr.shape, "| platform:", sorted(gse_pheno["Sample_platform_id"].unique()))

In [ ]:
def find_col(pheno, pattern):
    # Cot characteristics dau tien co gia tri khop pattern (vd 'disease_stage:')
    for c in pheno.columns:
        if pheno[c].astype(str).str.contains(pattern, case=False, na=False).any():
            return c
    return None


STAGE_COL = find_col(gse_pheno, r"disease_stage\s*:")
STATE_COL = find_col(gse_pheno, r"disease_state\s*:")
print("Cot stage:", STAGE_COL, "| cot disease_state:", STATE_COL)

ph = gse_pheno.copy()
ph["stage"] = ph[STAGE_COL].map(normalize_stage)
is_normal = ph[STATE_COL].astype(str).str.contains("normal", case=False, na=False)
ph.loc[is_normal, "stage"] = "Normal"

if not GSE_KEEP_REFERENCE_NORMAL:
    ref = ph["Sample_source_name_ch1"].astype(str).str.contains("stratagene", case=False)
    print(f"Bo {int(ref.sum())} mau RNA tham chieu Stratagene")
    ph = ph.loc[~ref]

bad = ph["stage"].isna() | ~ph["stage"].isin(STAGE_ORDER)
print(f"Loai {int(bad.sum())} khoi u khong xac dinh duoc giai doan:",
      ph.loc[bad, STAGE_COL].tolist())
ph = ph.loc[~bad]

print(ph["stage"].value_counts().reindex(STAGE_ORDER).dropna().astype(int).to_dict())

In [ ]:
# Thang do: MAS5 o thang tuyen tinh -> log2
e = gse_expr.loc[:, ph.index]
if np.nanmax(e.values) >= 100:
    e = np.log2(e.clip(lower=0) + 1)
    print("Da chuyen sang log2")

# Probe -> gene symbol tu GPL96.annot.gz
gpl = ph["Sample_platform_id"].iloc[0]
annot_url = f"https://ftp.ncbi.nlm.nih.gov/geo/platforms/{geo_stub(gpl)}/{gpl}/annot/{gpl}.annot.gz"
annot_path = download(annot_url, GSE_RAW)
lines, in_table = [], False
with gzip.open(annot_path, "rt", encoding="utf-8", errors="replace") as f:
    for line in f:
        if line.startswith("!platform_table_begin"):
            in_table = True
            continue
        if line.startswith("!platform_table_end"):
            break
        if in_table:
            lines.append(line)
annot = pd.read_csv(io.StringIO("".join(lines)), sep="\t", dtype=str)
sym = annot.set_index("ID")["Gene symbol"].str.strip()
sym = sym[sym.notna() & (sym != "") & ~sym.str.contains("///", regex=False)]
print(f"{gpl}: {len(sym)} probe anh xa duoc toi dung 1 gene")

e = e.dropna(axis=0, how="any")   # probe co NaN (neu co) bi bo
gse_gene = collapse_by_symbol(e, sym)
print("Ma tran gene:", gse_gene.shape, "(gene x mau)")

gse_data, gse_meta = build_table(gse_gene, ph["stage"],
                                 pd.Series(ph.index, index=ph.index))   # 1 mau = 1 benh nhan
gse_data.iloc[:3, list(range(5)) + [-1]]

## 4. TCGA-LUAD và TCGA-LUSC (GDC API, STAR - Counts)

Mỗi case (bệnh nhân, `case_id` dạng `TCGA-XX-XXXX`) có thể có cả mẫu u lẫn mẫu mô
thường, và đôi khi nhiều aliquot u. Giống notebook cũ, mỗi case giữ **1 mẫu u**
(ưu tiên Primary Tumor, không FFPE) cùng mọi mẫu mô thường. Khi chia fold,
`patient_id = case_id` nên cặp u–thường của cùng một bệnh nhân luôn ở cùng fold.

Giá trị biểu hiện: `log2(TPM + 1)`, chỉ gene `protein_coding`, Ensembl ID được
đổi sang `gene_name` (trùng tên → giữ gene có phương sai lớn nhất).

In [ ]:
PROJECTS = ["TCGA-LUAD", "TCGA-LUSC"]
API = "https://api.gdc.cancer.gov"
BATCH = 60   # so file tai moi lan; giam neu bi timeout


def _post(url, payload, timeout=600, retries=4, raw=False):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    last = None
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.read() if raw else json.loads(r.read().decode())
        except Exception as ex:
            last = ex
            wait = 5 * (attempt + 1)
            print(f"    loi ({ex}), thu lai sau {wait}s ...")
            time.sleep(wait)
    raise RuntimeError(f"That bai sau {retries} lan: {last}")


FIELDS = ",".join([
    "file_id", "file_name", "cases.submitter_id",
    "cases.samples.submitter_id", "cases.samples.sample_type", "cases.samples.is_ffpe",
    "cases.diagnoses.ajcc_pathologic_stage",
])


def query_files(project):
    filters = {"op": "and", "content": [
        {"op": "in", "content": {"field": "cases.project.project_id", "value": [project]}},
        {"op": "in", "content": {"field": "data_category", "value": ["Transcriptome Profiling"]}},
        {"op": "in", "content": {"field": "data_type", "value": ["Gene Expression Quantification"]}},
        {"op": "in", "content": {"field": "analysis.workflow_type", "value": ["STAR - Counts"]}},
    ]}
    res = _post(f"{API}/files", {"filters": filters, "fields": FIELDS,
                                 "format": "JSON", "size": "20000"})
    return res["data"]["hits"]


def flatten(hits):
    rows = []
    for h in hits:
        case = (h.get("cases") or [{}])[0]
        samp = (case.get("samples") or [{}])[0]
        diag = (case.get("diagnoses") or [{}])[0]
        rows.append({
            "file_id": h["file_id"],
            "case_id": case.get("submitter_id"),
            "sample_id": samp.get("submitter_id"),
            "sample_type": samp.get("sample_type"),
            "is_ffpe": samp.get("is_ffpe"),
            "stage_raw": diag.get("ajcc_pathologic_stage"),
        })
    return pd.DataFrame(rows)


def make_batches(ids, batch_size):
    # GDC /data tra ve file THO (khong phai tar.gz) neu lo chi co 1 file_id
    # -> chia deu sao cho khong lo nao chi con 1 phan tu.
    n = len(ids)
    if n == 0:
        return []
    n_batches = (n + batch_size - 1) // batch_size
    if n_batches >= 2 and n % batch_size == 1:
        n_batches -= 1
    batches, start = [], 0
    for i in range(n_batches):
        size = -(-(n - start) // (n_batches - i))
        batches.append(ids[start:start + size])
        start += size
    return batches


def download_batch(file_ids, dest):
    # Cache chi hop le neu dung la gzip (magic bytes \x1f\x8b)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        with open(dest, "rb") as f:
            if f.read(2) == b"\x1f\x8b":
                return dest
        print(f"    {os.path.basename(dest)} khong phai gzip hop le -- tai lai")
    blob = _post(f"{API}/data", {"ids": file_ids}, raw=True)
    with open(dest, "wb") as f:
        f.write(blob)
    return dest


def assemble(project, manifest):
    # Doc cac tar.gz -> (tpm: ensembl x sample, gene_meta)
    mf = manifest.set_index("file_id")
    pdir = os.path.join(RAWDIR, project)
    tpm, gene_meta = {}, None
    for t in sorted(f for f in os.listdir(pdir) if f.endswith(".tar.gz")):
        with tarfile.open(os.path.join(pdir, t), "r:gz") as tar:
            for m in tar.getmembers():
                if not m.isfile() or not m.name.endswith(".tsv"):
                    continue
                fid = m.name.split("/")[0]
                if fid not in mf.index:
                    continue
                df = pd.read_csv(io.BytesIO(tar.extractfile(m).read()), sep="\t",
                                 comment="#", low_memory=False)
                df = df[~df["gene_id"].astype(str).str.startswith("N_")].set_index("gene_id")
                if gene_meta is None:
                    gene_meta = df[["gene_name", "gene_type"]].copy()
                tpm[mf.loc[fid, "sample_id"]] = df["tpm_unstranded"].astype(np.float32)
    return pd.DataFrame(tpm), gene_meta

In [ ]:
tcga_raw = {}
for p in PROJECTS:
    print(f"\n=== {p} ===")
    mf = flatten(query_files(p))
    print(f"  {len(mf)} file |", mf["sample_type"].value_counts().to_dict())

    pdir = os.path.join(RAWDIR, p)
    os.makedirs(pdir, exist_ok=True)
    batches = make_batches(mf["file_id"].tolist(), BATCH)
    for i, chunk in enumerate(batches):
        dest = os.path.join(pdir, f"batch_{i:03d}.tar.gz")
        print(f"  lo {i+1}/{len(batches)} ({len(chunk)} file) ...", end=" ", flush=True)
        download_batch(chunk, dest)
        print(f"{os.path.getsize(dest)/1e6:.1f} MB")

    tpm, gene_meta = assemble(p, mf)
    print(f"  TPM: {tpm.shape[0]} gene x {tpm.shape[1]} mau")
    tcga_raw[p] = {"manifest": mf, "tpm": tpm, "gene_meta": gene_meta}

In [ ]:
def label_and_dedup(mf, sample_ids):
    # Gan stage, loai u khong ro giai doan, giu 1 mau u / case + moi mau thuong.
    mf = mf.drop_duplicates("sample_id").set_index("sample_id")
    mf = mf.loc[mf.index.isin(sample_ids)].copy()
    st = mf["sample_type"].astype(str)
    mf["stage"] = mf["stage_raw"].map(normalize_stage)
    is_normal = st.str.contains("Normal", na=False)
    mf.loc[is_normal, "stage"] = "Normal"   # mo thuong LUON la Normal

    bad = mf["stage"].isna() | ~mf["stage"].isin(STAGE_ORDER)
    print(f"  loai {int(bad.sum())} mau u khong xac dinh duoc giai doan")
    mf = mf.loc[~bad]

    tum = mf[~mf["sample_type"].astype(str).str.contains("Normal", na=False)].copy()
    tum["primary_rank"] = (~tum["sample_type"].astype(str).eq("Primary Tumor")).astype(int)
    tum["ffpe_rank"] = tum["is_ffpe"].fillna(False).astype(bool).astype(int)
    tum = tum.sort_values(["case_id", "primary_rank", "ffpe_rank"])
    tum = tum[~tum["case_id"].duplicated()]
    nor = mf[mf["sample_type"].astype(str).str.contains("Normal", na=False)]
    out = pd.concat([tum, nor])
    print(f"  {len(mf)} -> {len(out)} mau sau khu trung lap (1 u / case)")
    return out


tcga_tables = {}
for p in PROJECTS:
    print(f"\n=== {p} ===")
    d = tcga_raw[p]
    ph_t = label_and_dedup(d["manifest"], d["tpm"].columns)
    tpm = d["tpm"].loc[:, ph_t.index]

    gm = d["gene_meta"].loc[tpm.index]
    pc = gm["gene_type"] == "protein_coding"
    expr = np.log2(tpm.loc[pc] + 1)
    gene = collapse_by_symbol(expr, gm.loc[pc, "gene_name"])
    print(f"  ma tran gene: {gene.shape} (protein_coding, log2(TPM+1))")

    data, meta = build_table(gene, ph_t["stage"], ph_t["case_id"])
    tcga_tables[p] = (data, meta)
    print(" ", meta["stage"].value_counts().reindex(STAGE_ORDER).dropna().astype(int).to_dict())

## 5. Chia 5 phần cân bằng lớp theo nhóm bệnh nhân, xoay vòng thành 5 fold

`StratifiedGroupKFold` chia dữ liệu thành 5 phần P1–P5: mỗi bệnh nhân chỉ ở đúng
1 phần, tỉ lệ lớp mỗi phần gần với toàn bộ dữ liệu. Vì kết quả phụ thuộc seed,
thử `SEED_SEARCH` seed và chọn cách chia có tổng độ lệch số mẫu mỗi lớp so với
mức lý tưởng (`tổng lớp / 5`) nhỏ nhất.

Sau đó ghép theo `FOLD_PLAN` thành 5 fold (train 3 phần / val 1 / test 1) và
kiểm tra bằng `assert`: trong từng fold, tập bệnh nhân của train, val, test đôi
một rời nhau; qua 5 fold, mỗi mẫu làm test đúng 1 lần và val đúng 1 lần.

In [ ]:
def make_parts(meta, n_splits=N_FOLDS, seeds=SEED_SEARCH):
    # Tra ve so phan (1..n_splits) cua tung mau
    y = meta["label"].values
    groups = meta["patient_id"].values
    ideal = pd.Series(y).value_counts().sort_index() / n_splits
    best_part, best_score, best_seed = None, np.inf, None
    for s in seeds:
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=s)
        part = np.empty(len(y), dtype=int)
        for k, (_, idx) in enumerate(sgkf.split(np.zeros(len(y)), y, groups), start=1):
            part[idx] = k
        ct = pd.crosstab(part, y).reindex(columns=ideal.index, fill_value=0)
        score = (ct - ideal.values).abs().values.sum()
        if score < best_score:
            best_part, best_score, best_seed = part, score, s
    print(f"  seed chon: {best_seed} | tong do lech so mau/lop: {best_score:.1f}")
    return best_part


def fold_split(meta, r):
    # Vai tro (train/val/test) cua tung mau trong fold r
    plan = FOLD_PLAN[r]
    role = {p: "train" for p in plan["train"]}
    role[plan["val"]], role[plan["test"]] = "val", "test"
    return meta["part"].map(role)


def verify_no_patient_leak(meta):
    assert (meta.groupby("patient_id")["part"].nunique() == 1).all(), "Co benh nhan nam o nhieu phan"
    for r in FOLD_PLAN:
        split = meta[f"fold_{r}"]
        assert split.isin(["train", "val", "test"]).all()
        pats = {sp: set(meta.loc[split == sp, "patient_id"]) for sp in ["train", "val", "test"]}
        for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
            inter = pats[a] & pats[b]
            assert not inter, f"Fold {r}: benh nhan o ca {a} va {b}: {sorted(inter)[:5]}"
    # qua 5 fold, moi mau lam test dung 1 lan va val dung 1 lan
    roles = meta[[f"fold_{r}" for r in FOLD_PLAN]]
    assert ((roles == "test").sum(axis=1) == 1).all() and ((roles == "val").sum(axis=1) == 1).all()
    n_multi = int((meta.groupby("patient_id").size() > 1).sum())
    print(f"  OK: {meta['patient_id'].nunique()} benh nhan, 5 fold khong ro ri giua train/val/test "
          f"({n_multi} benh nhan co >1 mau, deu nam tron trong 1 phan)")


def fold_report(meta):
    code_to_stage = {v: k for k, v in STAGE_TO_CODE.items()}
    stages = meta["label"].map(code_to_stage)
    rows = []
    for r in FOLD_PLAN:
        ct = pd.crosstab(meta[f"fold_{r}"], stages).reindex(["train", "val", "test"])
        ct.index = pd.MultiIndex.from_product([[r], ct.index], names=["fold", "split"])
        rows.append(ct)
    ct = pd.concat(rows).fillna(0).astype(int)
    ct = ct.reindex(columns=[s for s in STAGE_ORDER if s in ct.columns])
    ct["tong"] = ct.sum(axis=1)
    return ct

## 6. Lưu 5 fold dạng H5 + CSV và đóng thành 2 file zip cho mỗi bộ

Mỗi fold là 1 thư mục `fold_<r>/` gồm `train`, `val`, `test` (H5 dùng key `data`).
Đọc lại: `pd.read_hdf("fold_1/train.h5", key="data")` hoặc
`pd.read_csv("fold_1/train.csv", index_col="sample_id")`.

Lưu ý dung lượng: mỗi mẫu có mặt trong cả 5 fold (3 lần ở train, 1 lần ở val,
1 lần ở test), nên zip lớn gấp khoảng 5 lần dữ liệu gốc.

In [ ]:
LABEL_MAPPING = {"label_to_stage": {v: k for k, v in STAGE_TO_CODE.items()},
                 "fold_plan": {f"fold_{r}": {"train_parts": p["train"], "val_part": p["val"],
                                             "test_part": p["test"]} for r, p in FOLD_PLAN.items()}}


def save_dataset(name, data, meta):
    print(f"\n=== {name}: {data.shape[0]} mau x {data.shape[1] - 1} gene ===")
    meta = meta.copy()
    meta["part"] = make_parts(meta)
    for r in FOLD_PLAN:
        meta[f"fold_{r}"] = fold_split(meta, r)
    verify_no_patient_leak(meta)
    display(fold_report(meta))

    ds_dir = os.path.join(WORKDIR, "folds_tmp", name)
    meta_csv = meta.to_csv()
    mapping_json = json.dumps(LABEL_MAPPING, indent=2)

    zips = []
    for fmt in ["h5", "csv"]:
        zip_path = os.path.join(OUTDIR, f"{name}_{fmt}.zip")
        with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for r in FOLD_PLAN:
                fold_dir = os.path.join(ds_dir, fmt, f"fold_{r}")
                os.makedirs(fold_dir, exist_ok=True)
                for split in ["train", "val", "test"]:
                    part = data.loc[meta.index[meta[f"fold_{r}"] == split]]
                    path = os.path.join(fold_dir, f"{split}.{fmt}")
                    if fmt == "h5":
                        part.to_hdf(path, key="data", mode="w", format="fixed")
                    else:
                        part.to_csv(path)
                    zf.write(path, arcname=f"{name}/fold_{r}/{split}.{fmt}")
                    os.remove(path)   # tiet kiem dia, ban chinh nam trong zip
            zf.writestr(f"{name}/fold_assignment.csv", meta_csv)
            zf.writestr(f"{name}/label_mapping.json", mapping_json)
        zips.append(zip_path)
        print(f"  -> {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
    return zips


DATASETS = {
    "gse68465": (gse_data, gse_meta),
    "tcga_luad": tcga_tables["TCGA-LUAD"],
    "tcga_lusc": tcga_tables["TCGA-LUSC"],
}

all_zips = []
for name, (data, meta) in DATASETS.items():
    all_zips += save_dataset(name, data, meta)

## 7. Kiểm tra lại 6 file zip

In [ ]:
assert len(all_zips) == 6
for zp in all_zips:
    with zipfile.ZipFile(zp) as zf:
        names = zf.namelist()
        fold_files = sorted(n for n in names if "/fold_" in n and "/fold_assignment" not in n)
        n_folds = len({n.split("/")[1] for n in fold_files})
        # doc lai train cua fold 1 de chac chan co cot label va ten gene
        first = next(n for n in fold_files if n.split("/")[1] == "fold_1" and "/train." in n)
        with zf.open(first) as fh:
            if first.endswith(".csv"):
                df = pd.read_csv(fh, index_col="sample_id")
            else:
                tmp = os.path.join(WORKDIR, "_check.h5")
                with open(tmp, "wb") as out:
                    out.write(fh.read())
                df = pd.read_hdf(tmp, key="data")
    print(f"{os.path.basename(zp):22s} {n_folds} fold x {len(fold_files) // n_folds} file | "
          f"{'/'.join(first.split('/')[1:])}: {df.shape}, cot cuoi='{df.columns[-1]}', "
          f"vd gene={list(df.columns[:3])}")

## 8. Chạy thử Random Forest trên 3 bộ

Đọc lại fold từ file zip H5 vừa lưu (kiểm tra luôn file zip dùng được). Với mỗi
fold trong `RF_FOLDS`: huấn luyện trên `train`, đánh giá trên `val` và `test`
với một cấu hình Random Forest cố định, dùng toàn bộ gene (chưa chọn đặc trưng).
Vì dữ liệu lệch lớp, ngoài `classification_report` notebook in thêm
balanced accuracy và macro-F1. Đặt `RF_FOLDS = range(1, 6)` để chạy cả 5 fold
và xem trung bình ± độ lệch chuẩn.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             balanced_accuracy_score, f1_score, accuracy_score)

RF_FOLDS = [1]   # vd range(1, N_FOLDS + 1) de chay ca 5 fold

RF_PARAMS = dict(
    n_estimators=500,
    max_features="sqrt",
    min_samples_leaf=1,
    class_weight="balanced",   # du lieu lech lop (vd GSE68465: I chiem ~60%)
    n_jobs=-1,
    random_state=42,
)


def load_fold(name, r, fmt="h5"):
    # Doc fold r tu <OUTDIR>/<name>_<fmt>.zip -> {"train": df, "val": df, "test": df}
    zip_path = os.path.join(OUTDIR, f"{name}_{fmt}.zip")
    extract_dir = os.path.join(WORKDIR, "folds_check", f"{name}_{fmt}")
    out = {}
    with zipfile.ZipFile(zip_path) as zf:
        for split in ["train", "val", "test"]:
            path = zf.extract(f"{name}/fold_{r}/{split}.{fmt}", extract_dir)
            out[split] = (pd.read_hdf(path, key="data") if fmt == "h5"
                          else pd.read_csv(path, index_col="sample_id"))
    return out


def split_xy(df):
    return df.drop(columns="label"), df["label"].values

In [ ]:
code_to_stage = {v: k for k, v in STAGE_TO_CODE.items()}
rf_summary = []

for name in ["gse68465", "tcga_luad", "tcga_lusc"]:
    for r in RF_FOLDS:
        fold = load_fold(name, r, fmt="h5")
        train, val, test = fold["train"], fold["val"], fold["test"]
        assert not (set(train.index) & set(val.index) | set(train.index) & set(test.index)
                    | set(val.index) & set(test.index)), "Trung mau giua cac tap"

        X_tr, y_tr = split_xy(train)
        print("=" * 70)
        print(f"{name} / fold {r}: train {X_tr.shape}, val {val.shape[0]} mau, test {test.shape[0]} mau")

        t0 = time.time()
        rf = RandomForestClassifier(**RF_PARAMS).fit(X_tr, y_tr)
        print(f"Huan luyen xong trong {time.time() - t0:.1f}s")

        for split_name, df in [("val", val), ("test", test)]:
            X, y = split_xy(df)
            pred = rf.predict(X)
            labels = sorted(set(y) | set(pred))   # chi cac lop co mat, tranh dong support=0 lam lech macro avg
            names = [code_to_stage[l] for l in labels]
            print(f"\n--- {name} / fold {r} / {split_name} ---")
            print(classification_report(y, pred, labels=labels, target_names=names,
                                        digits=3, zero_division=0))
            cm = pd.DataFrame(confusion_matrix(y, pred, labels=labels),
                              index=[f"that_{n}" for n in names],
                              columns=[f"du_doan_{n}" for n in names])
            display(cm)
            rf_summary.append({
                "dataset": name, "fold": r, "split": split_name, "n": len(y),
                "accuracy": accuracy_score(y, pred),
                "balanced_acc": balanced_accuracy_score(y, pred),
                "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
                "weighted_f1": f1_score(y, pred, average="weighted", zero_division=0),
                # moc so sanh: accuracy khi luon doan lop dong nhat cua train
                "acc_doan_lop_dong_nhat": float(np.mean(y == pd.Series(y_tr).mode()[0])),
            })

In [ ]:
summary_df = pd.DataFrame(rf_summary).set_index(["dataset", "fold", "split"]).round(3)
display(summary_df)

if len(RF_FOLDS) > 1:
    metrics = ["accuracy", "balanced_acc", "macro_f1", "weighted_f1", "acc_doan_lop_dong_nhat"]
    agg = pd.DataFrame(rf_summary).groupby(["dataset", "split"])[metrics].agg(["mean", "std"])
    display(agg.round(3))

## 9. (Tuỳ chọn) Ngắt kết nối Colab

Chỉ chạy sau khi đã thấy đủ 6 file zip trên Drive.

In [ ]:
if IN_COLAB:
    print(os.listdir(OUTDIR))
    from google.colab import runtime
    runtime.unassign()